# 👵 Nursing Home Fall Detection - YOLO11 Training

Notebook ini digunakan untuk melatih dan mengeksplorasi hyperparameter model **YOLO11** pada dataset **UR Fall Detection**.

### 📌 Petunjuk Penggunaan di Kaggle:
1. Nyalakan GPU di menu: **Settings -> Accelerator -> GPU T4 x1**.
2. Pastikan dataset sudah ditambahkan via menu: **+ Add Data** -> pilih dataset `urfall-yolo-dataset`.
3. Jalankan cell secara berurutan dari atas ke bawah.

### 1. Install Library & Cek GPU

In [ ]:
!pip install -q ultralytics

import torch
import yaml
import zipfile
from pathlib import Path
from ultralytics import YOLO

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name     :', torch.cuda.get_device_name(0))


### 2. Ekstrak Dataset di Workspace Kaggle

In [ ]:
kaggle_input = Path('/kaggle/input')
work_dir = Path('/kaggle/working')
dataset_dir = work_dir / 'dataset'

# Cari file zip dataset di input Kaggle
zip_candidates = list(kaggle_input.glob('*/*.zip')) + list(kaggle_input.glob('*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f'Mengekstrak {zip_path} ke {dataset_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dataset_dir)
    print('✅ Ekstraksi dataset selesai!')
else:
    possible_dir = list(kaggle_input.glob('*/images'))
    if possible_dir:
        dataset_dir = possible_dir[0].parent
        print(f'Dataset ditemukan di folder: {dataset_dir}')
    else:
        print('⚠️ Dataset zip tidak ditemukan, periksa /kaggle/input!')


### 3. Generate data.yaml untuk Lingkungan Kaggle

In [ ]:
data_yaml_content = {
    'path': str(dataset_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {
        0: 'normal',
        1: 'transitional',
        2: 'lying_on_ground'
    }
}

kaggle_yaml_path = work_dir / 'data.yaml'
with open(kaggle_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'✓ data.yaml berhasil dibuat di: {kaggle_yaml_path}')
print(open(kaggle_yaml_path).read())


### 4. Training Model YOLO11

> 💡 **TIPS UNTUK TEMAN KELOMPOK (EKSPLORASI PARAMETER):**
> - **Varian Model**: ganti `model_name = 'yolo11s.pt'` atau `'yolo11m.pt'`.
> - **Optimizer**: ganti `optimizer='AdamW'` dan coba `lr0=0.001`.
> - **Epochs**: coba naikkan `epochs=70` atau `100` dengan `patience=30`.

In [ ]:
# Pilihan model: yolo11n.pt (nano), yolo11s.pt (small), yolo11m.pt (medium)
model_name = 'yolo11n.pt'
model = YOLO(model_name)

results = model.train(
    data=str(kaggle_yaml_path),
    epochs=50,
    batch=16,
    imgsz=640,
    patience=15,
    lr0=0.01,
    lrf=0.01,
    optimizer='auto',       # bisa dicoba: 'AdamW'
    flipud=0.0,            # vertikal flip mati (orientasi gravitasi lantai)
    fliplr=0.5,            # horizontal flip
    mosaic=1.0,
    project='/kaggle/working/models',
    name='fall_detection_experiment',
    exist_ok=True,
    save=True
)

print('\n🎉 Training selesai!')
print(f'Hasil disimpan di: {results.save_dir}')


### 5. Evaluasi pada Test Set

In [ ]:
metrics = model.val(data=str(kaggle_yaml_path), split='test')

print('\n' + '='*45)
print('📊 HASIL METRIK TEST SET AKHIR:')
print('='*45)
print(f'• Test mAP50    : {metrics.box.map50 * 100:.2f}%')
print(f'• Test mAP50-95 : {metrics.box.map * 100:.2f}%')
print(f'• Test Precision: {metrics.box.mp * 100:.2f}%')
print(f'• Test Recall   : {metrics.box.mr * 100:.2f}%')
print('='*45)
print('\n📁 Model terbaik kalian tersimpan di:')
print(f'{results.save_dir}/weights/best.pt')
print('Silakan download file best.pt ini dari tab Output Kaggle!')
